In [1]:
import sys
# adjust path
sys.path.append('../../../NER-german-telegram')

from src.helpers.db_helpers import execute_sql_select
import src.config.db_credentials as db

from datetime import datetime
from random import sample

import spacy
from spacy import displacy

## Data

In [ ]:
table_name = "telegram_seeds"
channel_name = 'Demotermine'
ts_before = datetime.now()
query = f"""SELECT * FROM {table_name} WHERE channel_name = '{channel_name}'"""
data = execute_sql_select(command=query, database=db.DB_NAME_TELEGRAM, return_result_as_df=True)
ts_after = datetime.now()

print(f"Took {ts_after - ts_before}")

In [ ]:
data.head()

In [5]:
df = data[['id', 'text']]

In [6]:
texts = df.text.sample(5, random_state=2)

## Spacy

In [7]:
nlp = spacy.load("de_core_news_sm")

In [8]:
for text in texts: 
    
    if text: 
        doc = nlp(text)
    
        if doc.ents:
            for ent in doc.ents:
                pass
                # print(ent.text, ent.label_)

In [9]:
for text in texts: 
    
    try: 
        
        doc = nlp(text)
        sentence_spans = list(doc.sents)
        displacy.render(sentence_spans, style="ent")
        
    except:
        pass

/Users/elisabeth/repos/NER-german-telegram/venv/lib/python3.9/site-packages/spacy/displacy/__init__.py:205: UserWarning: [W006] No entities to visualize found in Doc object. If this is surprising to you, make sure the Doc was processed using a model that supports named entity recognition, and check the `doc.ents` property manually if necessary.
  warnings.warn(Warnings.W006)


## Flair (default)

In [10]:
from flair.data import Sentence
from flair.models import SequenceTagger

# load tagger
tagger_default = SequenceTagger.load("flair/ner-german")

2022-05-24 10:02:39,891 loading file /Users/elisabeth/.flair/models/ner-german/a125be40445295f7e94d0afdb742cc9ac40ec4e93259dc30f35220ffad9bf1f6.f46c4c5cfa5e34baa838983373e30051cd1cf1e933499408a49e451e784b0a11
2022-05-24 10:02:55,172 SequenceTagger predicts: Dictionary with 20 tags: <unk>, O, B-PER, E-PER, S-LOC, B-MISC, I-MISC, E-MISC, S-PER, B-ORG, E-ORG, S-ORG, I-ORG, B-LOC, E-LOC, S-MISC, I-PER, I-LOC, <START>, <STOP>


In [11]:
tagger_lg = SequenceTagger.load("flair/ner-german-large")

2022-05-24 10:02:55,959 loading file /Users/elisabeth/.flair/models/ner-german-large/6b8de9edd73722050be2547acf64c037b2df833c6e8f0e88934de08385e26c1e.4b0797effcc6ebb1889d5d29784b97f0a099c1569b319d87d7c387e44e2bba48
2022-05-24 10:03:33,226 SequenceTagger predicts: Dictionary with 20 tags: <unk>, O, B-PER, E-PER, S-LOC, B-MISC, I-MISC, E-MISC, S-PER, B-ORG, E-ORG, S-ORG, I-ORG, B-LOC, E-LOC, S-MISC, I-PER, I-LOC, <START>, <STOP>


In [12]:
def get_ents(tagger, text):

    sentence = Sentence(text)

    # predict NER tags
    tagger.predict(sentence)

    # print sentence
    print(sentence)

    # print predicted NER spans
    print('The following NER tags are found:')
    # iterate over entities and print
    for entity in sentence.get_spans('ner'):
        print(entity)

In [13]:
for t in texts:
    
    try:
        get_ents(tagger_default, t)
    except:
        pass

Sentence: "# Baunatal # HE # Do0411 # Do1111 # Do1811 # Do2511 Raus auf die Straßen @ Demotermine ! 👉 Übersicht / Overview 👈" → ["Baunatal"/LOC]
The following NER tags are found:
Span[1:2]: "Baunatal" → LOC (0.9824)
Sentence: "Zeitz Schützenplatz @ GesichtZeitz @ SachsenAnhaltInfoChat 01042021 Raus auf die Straße @ Demotermine !" → ["Zeitz Schützenplatz"/ORG, "SachsenAnhaltInfoChat"/ORG]
The following NER tags are found:
Span[0:2]: "Zeitz Schützenplatz" → ORG (0.5771)
Span[5:6]: "SachsenAnhaltInfoChat" → ORG (0.743)
Sentence: "🌏 WORLD WIDE DEMONSTRATION FOR FREEDOM 5.0 🌎 💫 Indiana , Crown Point is rising up ! 💫 📅 Saturday 20th November , 12 - 2 PM 🏢 at the old Courthouse around the Square United with more than 40 countries and 150 cities around the world . # wewillALLbethere We are standing side by side for freedom , peace and human rights . Bigger and better than ever before . Together , We are Free . 📣 t.me / worldwidedemoUSAofficial 📣🌏 t.me / WorldWideDemonstration 👥🌏 t.me / WorldWi

In [14]:
for t in texts:
    
    try:
        get_ents(tagger_lg, t)
    except:
        pass

Sentence: "# Baunatal # HE # Do0411 # Do1111 # Do1811 # Do2511 Raus auf die Straßen @ Demotermine ! 👉 Übersicht / Overview 👈" → ["Baunatal"/LOC, "HE"/ORG]
The following NER tags are found:
Span[1:2]: "Baunatal" → LOC (1.0)
Span[3:4]: "HE" → ORG (0.9671)
Sentence: "Zeitz Schützenplatz @ GesichtZeitz @ SachsenAnhaltInfoChat 01042021 Raus auf die Straße @ Demotermine !" → ["Zeitz Schützenplatz"/LOC, "GesichtZeitz"/ORG, "SachsenAnhaltInfoChat"/LOC]
The following NER tags are found:
Span[0:2]: "Zeitz Schützenplatz" → LOC (0.6245)
Span[3:4]: "GesichtZeitz" → ORG (0.9997)
Span[5:6]: "SachsenAnhaltInfoChat" → LOC (0.9997)
Sentence: "🌏 WORLD WIDE DEMONSTRATION FOR FREEDOM 5.0 🌎 💫 Indiana , Crown Point is rising up ! 💫 📅 Saturday 20th November , 12 - 2 PM 🏢 at the old Courthouse around the Square United with more than 40 countries and 150 cities around the world . # wewillALLbethere We are standing side by side for freedom , peace and human rights . Bigger and better than ever before . Together 